In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import re

# 경로 설정
csv_path = Path("../dataset/데이터셋/DB테이블/채권선도.csv")
# json_dir = Path("../dataset/처리/채권선도/ocr_json")
json_dir = Path("../dataset/처리/채권선도/ocr_json_preprocessing")

# CSV 로드
df = pd.read_csv(csv_path, encoding="cp949")
df["종목코드"] = df["종목코드"].astype(str)

# 비교 대상 필드
fields_to_compare = [
    "Position", "액면 통화", "액면 금액", "기초자산", "효력발생일", "만기일",
    "BDC", "영업일적용도시", "정산방식",
    "만기정산금액", "선도가격"
]

# 정규화 함수
def normalize_value(val):
    val = str(val).strip()
    date_formats = ["%Y%m%d", "%Y-%m-%d", "%Y.%m.%d"]
    for fmt in date_formats:
        try:
            return datetime.strptime(val, fmt).date()
        except ValueError:
            continue
    try:
        return float(val)
    except (ValueError, TypeError):
        return val.lower()

# 정확도 계산 함수
def calculate_accuracy_and_errors(csv_row, json_data):
    correct = 0
    errors = {}
    field_hits = {}

    for field in fields_to_compare:
        csv_val = csv_row.get(field, "")
        json_val = json_data.get(field, "")

        # 추가 로직: CSV 값이 비어있고 JSON이 "NA"인 경우 정답 처리
        if (csv_val == "" or pd.isna(csv_val)) and str(json_val).strip().upper() == "NA":
            correct += 1
            field_hits[field] = 1
            continue

        norm_csv = normalize_value(csv_val)
        norm_json = normalize_value(json_val)

        if norm_csv == norm_json:
            correct += 1
            field_hits[field] = 1
        else:
            errors[field] = {"CSV": csv_val, "JSON": json_val}
            field_hits[field] = 0

    accuracy = correct / len(fields_to_compare)
    return accuracy, errors, field_hits

# 종목코드 추출 함수 (파일명에서)
def extract_stock_code(fn: str) -> str:
    m = re.match(r'^(\d+)', fn)
    return m.group(1) if m else ''

# 한글 폰트 설정
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

# 분석 시작
print(f"폴더 분석: {json_dir}\n{'-'*50}")

results = []
field_accuracies = defaultdict(list)
field_errors = defaultdict(list)

for json_file in json_dir.glob("*.json"):
    # extract_stock_code 함수를 사용하여 파일명에서 종목코드 추출
    extracted_code = extract_stock_code(json_file.name)
    
    matched_row = None
    matched_code = None

    # 추출된 종목코드로 먼저 매칭 시도
    if extracted_code and extracted_code in df["종목코드"].values:
        matched_row = df[df["종목코드"] == extracted_code].iloc[0]
        matched_code = extracted_code
    else:
        # 기존 방식으로 fallback
        for code in df["종목코드"]:
            if str(code) in json_file.name:
                matched_row = df[df["종목코드"] == code].iloc[0]
                matched_code = code
                break

    if matched_row is None:
        print(f"[건너뜀] JSON 파일명에 매칭되는 종목코드가 없음: {json_file.name}")
        continue

    try:
        with open(json_file, "r", encoding="utf-8") as f:
            json_data = json.load(f)

        acc, errors, field_hits = calculate_accuracy_and_errors(matched_row, json_data)
        
        results.append((json_file.name, acc, field_hits))

        # 필드별 정확도 저장
        for field, hit in field_hits.items():
            field_accuracies[field].append(hit)
            
            # 오류케이스 저장
            if hit == 0:
                field_errors[field].append({
                    "파일명": json_file.name,
                    "종목코드": matched_code,
                    "CSV값": matched_row.get(field, ""),
                    "JSON값": json_data.get(field, "")
                })

    except Exception as e:
        print(f"[오류 발생] {json_file.name}: {e}")

# 1. 전체 정확도 및 필드별 정확도 시각화
accuracy_list = [item[1] for item in results]
overall_accuracy = sum(accuracy_list) / len(accuracy_list) if accuracy_list else 0

print(f"\n전체 평균 정확도: {overall_accuracy:.4f} ({overall_accuracy:.2%}) - 총 {len(results)}개 파일 분석")

# 필드별 평균 정확도 계산
field_accuracy_means = {
    field: sum(vals) / len(vals) if vals else 0.0
    for field, vals in field_accuracies.items()
}

# 필드별 정확도 DataFrame
field_acc_df = pd.DataFrame.from_dict(field_accuracy_means, orient='index', columns=['정확도'])
field_acc_df = field_acc_df.sort_values('정확도', ascending=False)

# 시각화
plt.figure(figsize=(12, 8))

# 필드별 정확도
field_acc_df.plot(kind='barh', legend=False)
plt.title("채권선도 필드별 평균 정확도")
plt.xlabel("정확도")
plt.xlim(0, 1)
plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

print("\n필드별 평균 정확도:")
display(field_acc_df.round(4))

In [ ]:
# 2. 필드별 오류케이스 전체 출력
print("\n\n필드별 오류케이스 상세 분석")
print("="*80)

# 필드별 오류 요약 테이블 생성
field_error_summary = []
for field in fields_to_compare:
    error_count = len(field_errors[field]) if field in field_errors else 0
    total_count = len(field_accuracies[field]) if field in field_accuracies else 0
    error_rate = error_count / total_count if total_count > 0 else 0
    accuracy = (total_count - error_count) / total_count if total_count > 0 else 0
    field_error_summary.append({
        "필드명": field,
        "총 파일수": total_count,
        "오류 건수": error_count,
        "정확 건수": total_count - error_count,
        "오류율": f"{error_rate:.2%}",
        "정확도": f"{accuracy:.2%}"
    })

# DataFrame으로 변환하고 오류율 기준으로 정렬
error_summary_df = pd.DataFrame(field_error_summary)
error_summary_df = error_summary_df.sort_values(by="오류 건수", ascending=False)

print("필드별 오류 요약 테이블:")
display(error_summary_df)

# 상세 오류케이스별로 테이블 출력
for field in fields_to_compare:
    if field in field_errors and field_errors[field]:
        error_count = len(field_errors[field])
        total_count = len(field_accuracies[field])
        error_rate = error_count / total_count if total_count > 0 else 0
        
        print(f"\n\n필드: {field} - 오류 건수: {error_count}/{total_count} ({error_rate:.2%})")
        print("-" * 80)
        
        # 오류케이스를 DataFrame으로 변환 (원본 값 유지)
        error_cases = []
        for i, error in enumerate(field_errors[field], 1):
            csv_val = error['CSV값']
            json_val = error['JSON값']
            
            # CSV 값 처리 (NaN 확인 후 변환)
            if pd.isna(csv_val):
                csv_val = "NaN"
            elif isinstance(csv_val, float):
                if csv_val == int(csv_val):
                    csv_val = str(int(csv_val))
                else:
                    csv_val = str(csv_val)
            else:
                csv_val = str(csv_val)
                
            # JSON 값 처리 (NaN 확인 후 변환)
            if pd.isna(json_val):
                json_val = "NaN"
            elif isinstance(json_val, float):
                if json_val == int(json_val):
                    json_val = str(int(json_val))
                else:
                    json_val = str(json_val)
            else:
                json_val = str(json_val)
            
            error_cases.append({
                "번호": i,
                "종목코드": error['종목코드'],
                "CSV값": csv_val,
                "JSON값": json_val
            })
        
        error_df = pd.DataFrame(error_cases)
        display(error_df)
    else:
        print(f"\n필드: {field} - 오류 없음")